In [0]:
import pandas as pd
import numpy as np
from time import perf_counter

df = pd.read_csv("orders.csv", parse_dates=["placed_at"])


# (1) row-wise tier assignment

def tier(row):
    if row["price_each"] < 50:
        return "low"
    if row["price_each"] < 150:
        return "mid"
    return "high"


start = perf_counter()
df["tier_slow"] = df.apply(tier, axis=1)
end = perf_counter()
tier_slow_time = end - start


start = perf_counter()

conditions = [
    df["price_each"] < 50,
    df["price_each"] < 150
]

choices = [
    "low",
    "mid"
]

df["tier_fast"] = np.select(
    conditions,
    choices,
    default="high"
)

end = perf_counter()
tier_fast_time = end - start


tier_equal = (df["tier_slow"] == df["tier_fast"]).all()
tier_speedup = tier_slow_time / tier_fast_time


print("CASE 1 — tier assignment")
print("Why slow: df.apply(..., axis=1) calls a Python function separately for every row.")
print(f"Slow time: {tier_slow_time:.6f} s")
print(f"Fast time: {tier_fast_time:.6f} s")
print(f"Equal: {tier_equal}")
print(f"Speedup: {tier_speedup:.2f}x")
print("-" * 60)


# (2) string op via apply

start = perf_counter()
df["country_lower_slow"] = df["country"].apply(lambda s: s.lower())
end = perf_counter()
country_slow_time = end - start


start = perf_counter()
df["country_lower_fast"] = df["country"].str.lower()
end = perf_counter()
country_fast_time = end - start


country_equal = (df["country_lower_slow"] == df["country_lower_fast"]).all()
country_speedup = country_slow_time / country_fast_time


print("CASE 2 — string lower")
print("Why slow: .apply(lambda s: s.lower()) executes a Python lambda for every text cell.")
print(f"Slow time: {country_slow_time:.6f} s")
print(f"Fast time: {country_fast_time:.6f} s")
print(f"Equal: {country_equal}")
print(f"Speedup: {country_speedup:.2f}x")
print("-" * 60)


# (3) building a column with a Python loop

start = perf_counter()

rev = []
for _, row in df.iterrows():
    rev.append(row["quantity"] * row["price_each"])

df["revenue_slow"] = rev

end = perf_counter()
revenue_slow_time = end - start


start = perf_counter()
df["revenue_fast"] = df["quantity"] * df["price_each"]
end = perf_counter()
revenue_fast_time = end - start


revenue_equal = np.allclose(df["revenue_slow"], df["revenue_fast"])
revenue_speedup = revenue_slow_time / revenue_fast_time


print("CASE 3 — revenue column")
print("Why slow: iterrows() loops through the DataFrame row by row in Python and builds a list manually.")
print(f"Slow time: {revenue_slow_time:.6f} s")
print(f"Fast time: {revenue_fast_time:.6f} s")
print(f"Equal: {revenue_equal}")
print(f"Speedup: {revenue_speedup:.2f}x")
print("-" * 60)
